In [ ]:
!pip install torch==2.6.0 torchvision torchaudio

INFO: pip is looking at multiple versions of torchvision to determine which version is compatible with other requirements. This could take a while.
INFO: pip is still looking at multiple versions of torchvision to determine which version is compatible with other requirements. This could take a while.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 766.6/766.6 MB 2.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 4.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 62.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.6/24.6 MB 59.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 883.7/883.7 kB 47.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 2.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.5/211.5 MB 7.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 MB 12.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 127.9/127.9 MB 8

In [1]:
import requests
from bs4 import BeautifulSoup

def inspect_html_structure(url: str, max_blocks: int = 50):
    headers = {"User-Agent": "Mozilla/5.0"}

    res = requests.get(url, headers=headers, timeout=15)
    res.raise_for_status()

    soup = BeautifulSoup(res.text, "html.parser")

    print("\n==============================")
    print("   HTML STRUCTURE + CONTENT   ")
    print("==============================\n")

    print("TITLE:", soup.title.get_text(strip=True) if soup.title else "No title")

    print("\n--- TOP LEVEL CONTENT BLOCKS ---\n")

    # focus on meaningful containers
    candidates = soup.find_all(["main", "article", "section", "div"], limit=max_blocks)

    count = 0

    for tag in candidates:
        text = tag.get_text(" ", strip=True)

        # skip empty/noisy blocks
        if len(text) < 80:
            continue

        # tag identity
        tag_name = tag.name
        tag_id = f"#{tag['id']}" if tag.get("id") else ""
        tag_class = "." + ".".join(tag.get("class")) if tag.get("class") else ""

        print("\n------------------------------")
        print(f"<{tag_name}{tag_id}{tag_class}>")
        print("TEXT SAMPLE:")
        print(text[:300])  # preview only

        count += 1
        if count >= max_blocks:
            break

    print("\n--- HEADINGS SNAPSHOT ---\n")
    for h in soup.find_all(["h1", "h2", "h3"])[:20]:
        print(f"{h.name}: {h.get_text(strip=True)}")
inspect_html_structure("https://www.nhs.uk/pregnancy/")


   HTML STRUCTURE + CONTENT   

TITLE: Stomach pain in pregnancy - NHS

--- TOP LEVEL CONTENT BLOCKS ---


------------------------------
<div.nhsuk-header__navigation-container.nhsuk-width-container>
TEXT SAMPLE:
Health A to Z NHS services Healthy living Mental health Care and support Browse More

------------------------------
<div.nhsuk-width-container>
TEXT SAMPLE:
Home Pregnancy Pregnancy symptoms Back to Pregnancy symptoms Back Stomach pain in pregnancy Stomach (abdominal) pains or cramps are common in pregnancy. They're usually nothing to worry about, but they can sometimes be a sign of something more serious that needs to be checked. It's probably nothing 

------------------------------
<main#maincontent.nhsuk-main-wrapper.nhsuk-main-wrapper--s>
TEXT SAMPLE:
Stomach pain in pregnancy Stomach (abdominal) pains or cramps are common in pregnancy. They're usually nothing to worry about, but they can sometimes be a sign of something more serious that needs to be checked. It's prob

In [52]:
import re
import time
import requests
from urllib.parse import urljoin, urlparse
from bs4 import BeautifulSoup

HEADERS = {"User-Agent": "Mozilla/5.0"}
DELAY   = 1.0   # polite crawl delay between requests (seconds)

# ── All NHS /pregnancy/ content URLs (pre-mapped, no JS required) ─────────────
SEED_URLS = [
    # ── Preparing for labour and birth (sub-pages) ────────────────────────────
    "https://www.nhs.uk/best-start-in-life/pregnancy/preparing-for-labour-and-birth/pain-relief-and-medication-during-labour/",
    "https://www.nhs.uk/best-start-in-life/pregnancy/preparing-for-labour-and-birth/signs-of-going-into-labour/",
    "https://www.nhs.uk/best-start-in-life/pregnancy/preparing-for-labour-and-birth/hospital-bag-checklist/",
    "https://www.nhs.uk/best-start-in-life/pregnancy/preparing-for-labour-and-birth/choosing-where-to-give-birth/",
    "https://www.nhs.uk/best-start-in-life/pregnancy/preparing-for-labour-and-birth/giving-birth/",
    "https://www.nhs.uk/best-start-in-life/pregnancy/preparing-for-labour-and-birth/how-to-use-a-birthing-ball/",
    "https://www.nhs.uk/best-start-in-life/pregnancy/preparing-for-labour-and-birth/antenatal-and-hypnobirthing-classes/",
    "https://www.nhs.uk/best-start-in-life/pregnancy/preparing-for-labour-and-birth/what-to-buy-for-your-newborn-baby/",
    "https://www.nhs.uk/best-start-in-life/pregnancy/preparing-for-labour-and-birth/what-to-include-in-your-birth-plan/",
    "https://www.nhs.uk/best-start-in-life/pregnancy/preparing-for-labour-and-birth/overdue-have-you-gone-past-your-due-date/",
    "https://www.nhs.uk/best-start-in-life/pregnancy/preparing-for-labour-and-birth/tips-for-your-birthing-partner-or-partners/",

    # ── Single-page articles (no sub-pages) ───────────────────────────────────
    "https://www.nhs.uk/best-start-in-life/pregnancy/healthy-eating-in-pregnancy/",
    "https://www.nhs.uk/best-start-in-life/pregnancy/vitamins-and-supplements-in-pregnancy/",
    "https://www.nhs.uk/best-start-in-life/pregnancy/smoking-and-alcohol-during-pregnancy/",
    "https://www.nhs.uk/best-start-in-life/pregnancy/exercising-in-pregnancy/",
    "https://www.nhs.uk/best-start-in-life/pregnancy/mental-health-and-pregnancy/",
    "https://www.nhs.uk/best-start-in-life/pregnancy/advice-for-partners/",
    "https://www.nhs.uk/best-start-in-life/pregnancy/morning-sickness/",
    "https://www.nhs.uk/best-start-in-life/pregnancy/using-hair-dye-in-pregnancy-is-it-safe/",
]

# ── Noise patterns to strip ───────────────────────────────────────────────────
NOISE_RE = re.compile(
    r'\b(learn more|read more|back to top|share this|expand all|collapse all)\b',
    flags=re.IGNORECASE
)

def clean_text(text: str) -> str:
    text = NOISE_RE.sub('', text)
    text = re.sub(r'\s{2,}', ' ', text)
    return text.strip()


# ── Section scraper ───────────────────────────────────────────────────────────
def scrape_sections(url: str) -> list[dict]:
    res = requests.get(url, headers=HEADERS, timeout=15)
    res.raise_for_status()
    soup = BeautifulSoup(res.text, "html.parser")
    article = soup.find("article") or soup.find("main") or soup.body

    sections           = []
    current_heading    = None
    current_level      = None
    current_body_parts = []

    def flush(heading, level, parts):
        if not heading or not parts:
            return
        body = clean_text(" ".join(parts))
        if len(body) < 20:
            return
        sections.append({"heading": clean_text(heading), "level": level, "body": body})

    BLOCK_TAGS = {"p", "li", "dd", "dt"}

    for tag in article.descendants:
        if not hasattr(tag, "name") or tag.name is None:
            continue
        if tag.name in ("h2", "h3"):
            flush(current_heading, current_level, current_body_parts)
            current_heading    = tag.get_text(" ", strip=True)
            current_level      = tag.name
            current_body_parts = []
            heading_text = tag.get_text(" ", strip=True)
    # Stop collecting at footer/noise sections
            STOP_HEADINGS = {"sign up for emails", "support links", "is this page useful", "thank you for your feedback"}
            if heading_text.lower() in STOP_HEADINGS:
              break
            flush(current_heading, current_level, current_body_parts)
            current_heading = heading_text
        elif tag.name in BLOCK_TAGS and current_heading:
          # Skip if this tag is nested inside another block tag (avoids double-counting)
          if tag.parent.name in BLOCK_TAGS:
            continue
          text = tag.get_text(" ", strip=True)
          if text and len(text) > 3:
            current_body_parts.append(text)
        elif tag.name == "div" and current_heading:
            classes = tag.get("class", [])
            if any(c in classes for c in ("nhsuk-inset-text", "nhsuk-card--care", "nhsuk-warning-callout")):
                card_text = tag.get_text(" ", strip=True)
                if card_text:
                    current_body_parts.append(card_text)

    flush(current_heading, current_level, current_body_parts)
    return sections


# ── Chunk builder ─────────────────────────────────────────────────────────────
def build_chunks(sections: list[dict], url: str, chunk_id_offset: int = 0) -> list[dict]:
    # Derive a section label from the URL path, e.g. "keeping-well / foods-to-avoid"
    parts = [p for p in urlparse(url).path.strip("/").split("/") if p and p != "pregnancy"]
    section_label = " / ".join(parts)

    return [
        {
            "id": f"nhs_section_{chunk_id_offset + i}",
            "content": s["body"],
            "metadata": {
                "source":        "nhs",
                "url":           url,
                "section":       section_label,
                "type":          "section",
                "heading":       s["heading"],
                "level":         s["level"],
            },
            "length": len(s["body"]),
        }
        for i, s in enumerate(sections)
    ]


# ── Main crawler ──────────────────────────────────────────────────────────────
def crawl() -> list[dict]:
    all_chunks   = []
    chunk_offset = 0
    total        = len(SEED_URLS)

    for i, url in enumerate(SEED_URLS, 1):
        print(f"📄 [{i}/{total}] {url}")
        try:
            sections = scrape_sections(url)
            chunks   = build_chunks(sections, url, chunk_id_offset=chunk_offset)
            all_chunks.extend(chunks)
            chunk_offset += len(chunks)
            print(f"   ✅ {len(chunks)} chunks")
        except Exception as e:
            print(f"   ⚠️  Failed: {e}")

        if i < total:
            time.sleep(DELAY)

    return all_chunks


# ── Run ───────────────────────────────────────────────────────────────────────
if __name__ == "__main__":
    all_chunks = crawl()

    lengths = [c["length"] for c in all_chunks]
    pages   = len({c["metadata"]["url"] for c in all_chunks})

    print("\n" + "=" * 60)
    print(f"Total pages  : {pages}")
    print(f"Total chunks : {len(all_chunks)}")
    if lengths:
        print(f"Min length   : {min(lengths)}")
        print(f"Max length   : {max(lengths)}")
        print(f"Mean length  : {sum(lengths) / len(lengths):.1f}")
    print()

    for c in all_chunks:
        print("─" * 60)
        print(f"[{c['metadata']['level'].upper()}] {c['metadata']['heading']}")
        print(f"  Section : {c['metadata']['section']}")
        print(f"  URL     : {c['metadata']['url']}")
        print(c["content"][:300], "..." if c["length"] > 300 else "")
        print()

📄 [1/19] https://www.nhs.uk/best-start-in-life/pregnancy/preparing-for-labour-and-birth/pain-relief-and-medication-during-labour/
   ✅ 8 chunks
📄 [2/19] https://www.nhs.uk/best-start-in-life/pregnancy/preparing-for-labour-and-birth/signs-of-going-into-labour/
   ✅ 6 chunks
📄 [3/19] https://www.nhs.uk/best-start-in-life/pregnancy/preparing-for-labour-and-birth/hospital-bag-checklist/
   ✅ 5 chunks
📄 [4/19] https://www.nhs.uk/best-start-in-life/pregnancy/preparing-for-labour-and-birth/choosing-where-to-give-birth/
   ✅ 6 chunks
📄 [5/19] https://www.nhs.uk/best-start-in-life/pregnancy/preparing-for-labour-and-birth/giving-birth/
   ✅ 5 chunks
📄 [6/19] https://www.nhs.uk/best-start-in-life/pregnancy/preparing-for-labour-and-birth/how-to-use-a-birthing-ball/
   ✅ 4 chunks
📄 [7/19] https://www.nhs.uk/best-start-in-life/pregnancy/preparing-for-labour-and-birth/antenatal-and-hypnobirthing-classes/
   ✅ 6 chunks
📄 [8/19] https://www.nhs.uk/best-start-in-life/pregnancy/preparing-for-labour-and-b

In [ ]:
import torch
print(torch.__version__)

2.11.0+cu128


In [53]:

print("Total chunks:", len(all_chunks))

Total chunks: 109


In [54]:
import statistics
from collections import Counter

# ── Safety check ───────────────────────────────
if not all_chunks:
    print("No chunks found.")
    exit()

# ── Size distribution ──────────────────────────
lengths = [len(c.get("content", "")) for c in all_chunks]

print(f"\n📏 Chunk size stats:")

if lengths:
    print(f"  Min    : {min(lengths)}")
    print(f"  Max    : {max(lengths)}")
    print(f"  Mean   : {statistics.mean(lengths):.1f}")
    print(f"  Median : {statistics.median(lengths):.1f}")
else:
    print("  No valid content lengths found")


# ── Source breakdown ───────────────────────────
sources = Counter(
    c.get("metadata", {}).get("source", "unknown")
    for c in all_chunks
)

print(f"\n📊 Chunk sources:")
for s, count in sources.most_common():
    print(f"  {s}: {count}")


# ── Type breakdown (VERY USEFUL for FAQ) ───────
types = Counter(
    c.get("metadata", {}).get("type", "unknown")
    for c in all_chunks
)

print(f"\n🏷️ Chunk types:")
for t, count in types.most_common():
    print(f"  {t}: {count}")


# ── URL check ──────────────────────────────────
urls = Counter(
    c.get("metadata", {}).get("url", "unknown")
    for c in all_chunks
)

print(f"\n🌐 URL distribution:")
for u, count in urls.most_common(5):
    print(f"  {u[:60]}... : {count}")


# ── Sample chunks ──────────────────────────────
print("\n📌 Sample chunks:")

for c in all_chunks[0:3]:

    meta = c.get("metadata", {})

    question = meta.get("question", "N/A")
    content  = c.get("content", "")

    print(f"\n  Question : {question[:80]}")
    print(f"  Source   : {meta.get('source', 'unknown')}")
    print(f"  Length   : {c.get('length', len(content))}")
    print(f"  Content  :\n{content}")
    print("  " + "-" * 50)


📏 Chunk size stats:
  Min    : 43
  Max    : 1773
  Mean   : 390.9
  Median : 294.0

📊 Chunk sources:
  nhs: 109

🏷️ Chunk types:
  section: 109

🌐 URL distribution:
  https://www.nhs.uk/best-start-in-life/pregnancy/exercising-i... : 13
  https://www.nhs.uk/best-start-in-life/pregnancy/healthy-eati... : 12
  https://www.nhs.uk/best-start-in-life/pregnancy/vitamins-and... : 11
  https://www.nhs.uk/best-start-in-life/pregnancy/preparing-fo... : 8
  https://www.nhs.uk/best-start-in-life/pregnancy/smoking-and-... : 7

📌 Sample chunks:

  Question : N/A
  Source   : nhs
  Length   : 85
  Content  :
TENS machine Epidural Gas and air Pethidine Self-help Alternative methods Water birth
  --------------------------------------------------

  Question : N/A
  Source   : nhs
  Length   : 369
  Content  :
A TENS machine delivers small amounts of electrical currents through pads on your back. It's believed to encourage the body to produce more of its own natural painkillers (endorphins). If you wa

In [7]:
!pip install FlagEmbedding

  Preparing metadata (setup.py) ... done
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 247.7/247.7 kB 12.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 866.1/866.1 kB 35.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 149.7/149.7 kB 17.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.6/45.6 kB 5.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.2/5.2 MB 100.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 70.4 MB/s eta 0:00:00
  Created wheel for warc3-wet-clueweb09: filename=warc3_wet_clueweb09-0.2.5-py3-none-any.whl size=18919 sha256=28691655f2f307f78facee8d93d56de9544e1ba2e652c599a9eeef93df0775e7
  Stored in directory: /root/.cache/pip/wheels/f6/85/c2/9f0f621def52a1d5db7d29984f81e45f9fb6dfeb1a4eb6e31c
  Created wheel for cbor: filename=cbor-1.0.0-cp312-cp312-linux_x86_64.whl size=55024 sha256=80ed42c27e64e485f5c08ff6e2fe7d1fc77705b1fb23a44837689800696ce903
 

In [55]:
import os
os.environ["TRANSFORMERS_SAFETENSORS_FAST"] = "1"
os.environ["USE_SAFETENSORS"] = "1"

from FlagEmbedding import BGEM3FlagModel
import numpy as np

model = BGEM3FlagModel("BAAI/bge-m3", use_fp16=True)  # load once


def embed_chunks(chunks, batch_size=32):
    """
    Embeds chunks using BGE-M3.
    Uses question as context prefix for better retrieval.
    """
    texts = []

    for c in chunks:
        content  = c.get("content", "")
        question = c.get("metadata", {}).get("question", "")

        # context-aware: question as semantic header
        combined = f"Question: {question}\n\n{content}"
        texts.append(combined)

    print(f"🔄 Embedding {len(texts)} chunks...")

    outputs = model.encode(
        texts,
        batch_size=batch_size,
        max_length=1024,
        return_dense=True
    )

    embeddings = outputs["dense_vecs"]

    embedded_chunks = []
    for i, c in enumerate(chunks):
        embedded_chunks.append({
            **c,
            "embedding": embeddings[i]
        })

    print(f"✅ Done. Embedding shape: {embeddings.shape}")
    return embedded_chunks, embeddings



Fetching 30 files:   0%|          | 0/30 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

In [56]:

embedded_chunks, embeddings = embed_chunks(all_chunks)  # ← blocks, not chunks_clean

🔄 Embedding 109 chunks...


Inference Embeddings: 100%|██████████| 4/4 [00:01<00:00,  3.34it/s]

✅ Done. Embedding shape: (109, 1024)


In [57]:
import hashlib

def build_metadata_chunks(chunks,
                          source_name="nhs-best-start-in-life",
                          source_org="NHS",
                          doc_id="nhs-best-start-in-life",
                          authority_level="Primary"):

    total = len(chunks)
    structured = []

    for i, chunk in enumerate(chunks):

        content  = chunk.get("content", "")
        metadata = chunk.get("metadata", {})

        if not content:
            print(f"[WARN] Empty content at index {i}, skipping.")
            continue

        uid = hashlib.md5(
            f"{doc_id}_{i}_{content[:50]}".encode()
        ).hexdigest()[:12]

        record = {
            "id": uid,
            "content": content,
            "embedding": chunk.get("embedding"),

            "metadata": {
              "source":          source_org,
              "page":            -1,  # no pages in FAQ
              "pub_year":        -1,
              "chunk_index":     i,
              "total_chunks":    total,
              "week":            False,    # no week in FAQ; PDFs will have "week-1" etc
              "section_title":   metadata.get("heading", ""),
              "chunk_source":    metadata.get("url", ""),
              "source_org":      source_org,
              "doc_id":          doc_id,
              "authority_level": authority_level,
}
        }

        structured.append(record)

    return structured




In [58]:
structured_chunks = build_metadata_chunks(
    embedded_chunks,
    source_name="nhs-best-start-in-life",
    source_org="NHS",
    doc_id="nhs-best-start-in-life",
    authority_level="Primary"
)

print(f"✅ Total structured chunks: {len(structured_chunks)}")

✅ Total structured chunks: 109


In [59]:
# Preview one
import json
sample = structured_chunks[4].copy()
sample["embedding"] = f"[vector dim {len(sample['embedding'])}]"  # don't print full vector
print(json.dumps(sample, indent=2))

{
  "id": "f2153543ad60",
  "content": "If you have pethidine in labour, it\u2019s given as an injection into your thigh or bottom. Pethidine: can help you relax for around 2 to 4 hours takes about 20 minutes to work is not suitable for late stages of labour as the drugs can affect the baby's breathing and feeding Pethidine can make you feel woozy, sick and forgetful.",
  "embedding": "[vector dim 1024]",
  "metadata": {
    "source": "NHS",
    "page": -1,
    "pub_year": -1,
    "chunk_index": 4,
    "total_chunks": 109,
    "week": false,
    "section_title": "Pethidine",
    "chunk_source": "https://www.nhs.uk/best-start-in-life/pregnancy/preparing-for-labour-and-birth/pain-relief-and-medication-during-labour/",
    "source_org": "NHS",
    "doc_id": "nhs-best-start-in-life",
    "authority_level": "Primary"
  }
}


In [60]:
import numpy as np

def get_vector(x):

    # FlagEmbedding M3 output (MOST IMPORTANT CASE)
    if isinstance(x, dict):

        if "dense_vecs" in x:
            return np.array(x["dense_vecs"])

        if "embedding" in x:
            return np.array(x["embedding"])


        if "data" in x:
            return np.array(x["data"])

    # list / raw vector
    if isinstance(x, list):
        return np.array(x)

    # already numpy
    return np.array(x)

In [61]:
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np

def search(query, embedded_chunks, model, top_k=3):

    # ---- query embedding ----
    q_raw = model.encode(query)
    q_vec = get_vector(q_raw).reshape(1, -1)

    results = []

    for chunk in embedded_chunks:

        c_vec = get_vector(chunk["embedding"]).reshape(1, -1)

        score = cosine_similarity(q_vec, c_vec)[0][0]

        results.append((score, chunk))

    return sorted(results, key=lambda x: x[0], reverse=True)[:top_k]

In [64]:
# ─────────────────────────────────────────────
# 🔎 QUERY TESTING (ENGLISH + BANGLA)
# ─────────────────────────────────────────────

queries = [
    "how to take care of my mind during pregnancy?",
    "প্রসব-পরবর্তী ব্যথা কীভাবে নিয়ন্ত্রণ করা যায় এবং কোন কোন উপায়ে এটি দ্রুত কমানো সম্ভব",
]

for q in queries:
    print("\n" + "="*90)
    print(f"🔍 QUERY: {q}")

    results = search(q, embedded_chunks, model, top_k=3)

    for score, chunk in results:
        meta = chunk.get("metadata", {})
        print(f"\n  Score    : {score:.4f}")
        print(f"  Question : {meta.get('question', 'N/A')}")
        print(f"  Source   : {meta.get('source', 'N/A')}")
        print(f"  Section   : {meta.get('heading', 'N/A')}")
        print(f"  Content  : {chunk.get('content', '')[:300]}")
        print("-" * 70)


🔍 QUERY: how to take care of my mind during pregnancy?

  Score    : 0.6582
  Question : N/A
  Source   : nhs
  Section   : Looking after yourself
  Content  : Just as you are planning your baby's arrival, you can also plan to look after yourself during your pregnancy. Tommy's Wellbeing Plan can help do this and be prepared for after the birth. If you'd like help and information on taking care of yourself specifically during pregnancy, have a look at Tommy
----------------------------------------------------------------------

  Score    : 0.6147
  Question : N/A
  Source   : nhs
  Section   : Share your worries
  Content  : It's normal to have some worries during pregnancy. If you're concerned about how you're feeling, you can always talk to your midwife or doctor. They might be able to reassure you or point you in the right direction of all the support that you need, without judgment. If you're anxious about your rela
-----------------------------------------------------------------

In [16]:
# NEW CELL 1: Install Pinecone
!pip install pinecone sentence-transformers -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.0/3.0 MB 48.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 225.0/225.0 kB 22.5 MB/s eta 0:00:00


In [17]:
from pinecone import Pinecone, ServerlessSpec

# Connect Pinecone
pc = Pinecone(api_key="pcsk_2X2Gyr_De54caT3PgyLKzBKURoA8SyRA8zA3fFkbjMA5fqZ5qkcCkBgZ3TPKVjXmJNjwGn")

# Main index name
index_name = "maternal-health"

# Create only once
if index_name not in pc.list_indexes().names():

    pc.create_index(
        name=index_name,
        dimension=1024,   # BGE-M3 dimension
        metric="cosine",
        spec=ServerlessSpec(
            cloud="aws",
            region="us-east-1"
        )
    )

print("✅ Main index ready")

✅ Main index ready


In [65]:
# Connect to existing index
index = pc.Index("maternal-health")

# Namespace for THIS P
namespace = "NHS_BEST_START_IN_LIFE"

print("✅ Namespace selected:", namespace)

✅ Namespace selected: NHS_BEST_START_IN_LIFE


In [66]:
import numpy as np

vectors = []

for c in structured_chunks:

    embedding = np.array(c["embedding"]).astype(float).tolist()

    metadata = c["metadata"].copy()

    # remove None values (Pinecone doesn't accept nulls)
    metadata = {k: v for k, v in metadata.items() if v is not None}

    # store content inside metadata for retrieval
    metadata["content"] = c["content"]

    vectors.append({
        "id":     c["id"],
        "values": embedding,
        "metadata": metadata
    })

print("✅ Vectors prepared:", len(vectors))



✅ Vectors prepared: 109


In [67]:
batch_size = 100

for i in range(0, len(vectors), batch_size):

    index.upsert(
        vectors=vectors[i:i+batch_size],
        namespace=namespace
    )

    print(f"✅ Uploaded batch {i//batch_size + 1}")

print("✅ Upload complete")

✅ Uploaded batch 1
✅ Uploaded batch 2
✅ Upload complete
